# Stochastic Lotka–Volterra conditioned on early observations and a prey boom

The stochastic Lotka–Volterra model describes a prey population $u$ and a predator population $v$, here evolving in log-space as

$$d\log u = \left(\alpha - \beta v - \tfrac{1}{2}\sigma^2\right)dt + \sigma\,dW_u, \qquad d\log v = \left(\delta u - \gamma - \tfrac{1}{2}\sigma^2\right)dt + \sigma\,dW_v.$$

This notebook conditions the paths on noisy observations of both species early on, on a prey boom (peak prey above $7$) that the prior rarely produces, and on both together. The two species are coupled through the dynamics, so a condition on prey alone also reshapes the predator. The figures reproduce the Lotka–Volterra row of Figure 4 in *LatentFlow: A General Framework for Conditioning Stochastic Processes* (Sharrock, Astfalck & Moss, 2026).

## 1. Setup

Imports and the configuration shared by every section: device and seed, the time grid, the number of sample paths and flow integration steps, the observation settings, the event thresholds and the plot colours. The time grid is defined at module level so that the map and the conditions can both read it.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from flow_inference.conditions import (
    BinaryCondition,
    EqualityCondition,
    ProductCondition,
)
from flow_inference.flows import GaussianIncrementFlow
from flow_inference.guidance import GuidanceMC
from flow_inference.maps import BaseMap
from flow_inference.sampler import IntegratorSampler

sys.path.insert(0, str(Path.cwd().parent))  # make examples/plotting.py importable
from matplotlib.lines import Line2D
from matplotlib.offsetbox import AnchoredOffsetbox, TextArea, VPacker
from plotting import as_numpy, plot_traces

device = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0
torch.manual_seed(SEED)

num_samples = 64  # paths per prior / guided batch
num_times = 60  # time points per path
t_final = 5.0  # time horizon
dt = t_final / (num_times - 1)
times = torch.linspace(0.0, t_final, num_times, device=device)
NUM_STEPS = 100  # flow integration steps

OBS_WINDOW = 1.25  # observations are taken on t <= OBS_WINDOW
OBS_NOISE = 0.05  # observation noise standard deviation
PEAK_PREY = 7.0  # the prey-boom threshold

CHANNELS = ["prey", "predator"]
COLORS = {"prey": "#2563eb", "predator": "#dc2626"}
EVENT_LINES = [  # components of the event, for the threshold lines and the event box
    {"text": f"peak prey > {PEAK_PREY:g}", "color": COLORS["prey"], "channel": "prey", "y": PEAK_PREY, "span": "path"},
]

Plotting helpers used by every figure: `plot_paths` draws the sample paths, `plot_observations` marks the observations as black dots, and `show_event` draws each event threshold as a dashed line with a box listing the event's components.

In [ ]:
def plot_paths(ax, samples, title, ylabel="population"):
    """Every sample path of every channel of (B, C, num_times) samples."""
    for channel, name in enumerate(CHANNELS):
        plot_traces(samples, x=times, channel=channel, color=COLORS[name], count=samples.shape[0],
                    alpha=0.4, linewidth=1.2, ax=ax)
    ax.set_title(title, pad=8)
    ax.set_xlabel("time")
    if ylabel is not None:
        ax.set_ylabel(ylabel)
    ax.set_ylim(0.0, 14.0)
    ax.grid(True, color="#e5e7eb", linewidth=0.8)
    ax.margins(x=0.01)


def channel_legend(ax, loc):
    handles = [Line2D([], [], color=COLORS[name], linewidth=2.0, label=name) for name in CHANNELS]
    ax.legend(handles=handles, loc=loc)


def show_event(ax, loc):
    threshold_lines({name: ax for name in CHANNELS})
    event_box(ax, loc)


def plot_observations(ax, observations, channels, mask, transform=lambda y: y):
    """Observed values as black dots."""
    for channel in channels:
        ax.scatter(as_numpy(times[mask]), as_numpy(transform(observations[channel, mask])),
                   color="black", edgecolor="white", linewidth=0.6, s=30, zorder=5)


def event_box(ax, loc):
    """The event's components as coloured lines of text in a white box, as in the paper."""
    lines = [TextArea(line["text"], textprops=dict(color=line["color"], fontsize=9)) for line in EVENT_LINES]
    box = AnchoredOffsetbox(loc=loc, child=VPacker(children=lines, align="left", pad=0, sep=2),
                            pad=0.4, borderpad=0.6, frameon=True)
    box.patch.set_boxstyle("round")
    box.patch.set_edgecolor("#9ca3af")
    ax.add_artist(box)


def threshold_lines(axes_by_channel):
    """A dashed line at every event threshold: across the whole path for extremes, over the
    final stretch for terminal conditions."""
    for line in EVENT_LINES:
        xmin = 0.0 if line["span"] == "path" else 0.85
        axes_by_channel[line["channel"]].axhline(line["y"], xmin=xmin, color=line["color"],
                                                 linestyle="--", linewidth=1.5, zorder=4)

## 2. The stochastic Lotka–Volterra model

The parameters are prey growth rate $\alpha = 1.1$, predation rate $\beta = 0.45$, predator growth per prey $\delta = 0.35$, predator death rate $\gamma = 0.75$ and noise level $\sigma = 0.08$, over the time horizon $[0, 5]$.

### 2.1 The map from increments to paths

The process is written as a deterministic map from iid standard normal increments $z$ to paths, so that a flow over $z$ gives a flow over paths.

`LotkaVolterraMap` takes a `(B, 2, num_times)` latent tensor. The first time slice perturbs the initial populations, $\log u_0 = \log 2 + 0.15\,z^u_0$ and $\log v_0 = \log 1 + 0.15\,z^v_0$, and the remaining slices supply the Brownian increments of an Euler–Maruyama step of size $\Delta t$ in log-space:

$$\log u_{k+1} = \log u_k + \left(\alpha - \beta v_k - \tfrac{1}{2}\sigma^2\right)\Delta t + \sigma\sqrt{\Delta t}\,z^u_{k+1}, \qquad \log v_{k+1} = \log v_k + \left(\delta u_k - \gamma - \tfrac{1}{2}\sigma^2\right)\Delta t + \sigma\sqrt{\Delta t}\,z^v_{k+1}.$$

Working in log-space keeps both populations positive. The map returns prey and predator as its two channels.

In [ ]:
class LotkaVolterraMap(BaseMap):
    """Coupled (prey, predator) Euler-Maruyama simulator in log-space."""

    def __init__(
        self,
        num_times: int,
        alpha: float = 1.10,
        beta: float = 0.45,
        delta: float = 0.35,
        gamma: float = 0.75,
        sigma: float = 0.08,
    ):
        super().__init__(
            latent_dim=num_times,
            output_shape=(num_times,),
            channels=2,
            device=device,
        )
        self.num_times = num_times
        self.alpha = alpha
        self.beta = beta
        self.delta = delta
        self.gamma = gamma
        self.sigma = sigma

    def _map(self, z: torch.Tensor) -> torch.Tensor:
        """z: (B, 2, num_times) iid N(0, 1) -> (B, 2, num_times) (prey, predator) paths."""
        B, C, N = z.shape
        if C != 2 or N != self.num_times:
            raise ValueError(f"expected latent shape (B, 2, {self.num_times}), got {tuple(z.shape)}")

        sigma = self.sigma
        sqrt_dt = dt ** 0.5

        log_prey = torch.log(torch.full((B,), 2.0, device=z.device, dtype=z.dtype)) + 0.15 * z[:, 0, 0]
        log_predator = torch.log(torch.full((B,), 1.0, device=z.device, dtype=z.dtype)) + 0.15 * z[:, 1, 0]
        prey_path = [log_prey.exp()]
        predator_path = [log_predator.exp()]

        for k in range(self.num_times - 1):
            prey = log_prey.exp()
            predator = log_predator.exp()
            log_prey = log_prey + (self.alpha - self.beta * predator - 0.5 * sigma**2) * dt + sigma * sqrt_dt * z[:, 0, k + 1]
            log_predator = log_predator + (self.delta * prey - self.gamma - 0.5 * sigma**2) * dt + sigma * sqrt_dt * z[:, 1, k + 1]
            prey_path.append(log_prey.exp())
            predator_path.append(log_predator.exp())

        return torch.stack([torch.stack(prey_path, dim=-1), torch.stack(predator_path, dim=-1)], dim=1)

### 2.2 Prior samples

`GaussianIncrementFlow` runs the flow over the Gaussian increments and pushes the result through the map; `IntegratorSampler` integrates it from noise. Without guidance this gives unconditioned (prior) paths.

In [ ]:
flow = GaussianIncrementFlow(map=LotkaVolterraMap(num_times), device=device)
sampler = IntegratorSampler(flow=flow)
prior_samples = sampler.sample(num_samples, num_steps=NUM_STEPS).detach()

fig, ax = plt.subplots(figsize=(6, 3.6))
plot_paths(ax, prior_samples, "Prior")
channel_legend(ax, loc="upper right")
plt.tight_layout()

## 3. The conditions

A condition is a log-likelihood $\log p(\text{condition} \mid x)$ on a sample path $x$; guidance steers the flow towards paths where it is high. This notebook uses one condition built from data and one built from an event.

### 3.1 Noisy population observations

One prior path serves as the ground truth. Both populations are observed at every other time point of the early window $t \le 1.25$, with Gaussian noise of standard deviation $0.05$.

In [ ]:
truth = prior_samples[1]
obs_mask = (times <= OBS_WINDOW) & (torch.arange(num_times, device=device) % 2 == 0)
obs_channels = torch.tensor([0, 1], device=device)  # both species
generator = torch.Generator(device=device).manual_seed(SEED)
observations = truth + OBS_NOISE * torch.randn(truth.shape, device=device, generator=generator)

`NoisyObservation` is an `EqualityCondition` on the residual between the observed channels of a path and the observations $y_{c,j}$ at the observed times $t_j$:

$$\log p(y \mid x) = -\sum_{c,j} \frac{\big(x_c(t_j) - y_{c,j}\big)^2}{2\sigma^2}.$$

With $\sigma$ set to the observation noise, $0.05$, this is exactly the Gaussian observation model, so $\sigma$ needs no tuning.

In [ ]:
class NoisyObservation(EqualityCondition):
    """Gaussian observations of `channels` at the masked times, with noise standard deviation `sigma`."""

    def __init__(self, observations: torch.Tensor, channels: torch.Tensor, mask: torch.Tensor, sigma: float):
        super().__init__(sigma=sigma)
        self.register_buffer("observations", observations)
        self.register_buffer("channels", channels)
        self.register_buffer("mask", mask)

    def residual(self, x: torch.Tensor) -> torch.Tensor:
        selected = x[:, self.channels][:, :, self.mask]
        target = self.observations[self.channels][:, self.mask]
        return selected - target[None, :, :]


data_condition = NoisyObservation(observations, obs_channels, obs_mask, sigma=OBS_NOISE)

### 3.2 A prey boom

The event is a prey boom: the prey population peaks above $7$. Its margin is

$$m(x) = \operatorname{smax}_\tau(u) - 7,$$

with $\tau = 0.05$.

Extremes over time are taken with a smooth (log-sum-exp) maximum, which is differentiable and overestimates the true maximum by at most $\tau \log N$ over $N$ time points:

$$\operatorname{smax}_\tau(x) = \tau \log \sum_{k} e^{x_k / \tau}.$$

A `BinaryCondition` turns margins $m_a(x)$, positive when component $a$ of the event holds, into the log-likelihood

$$\log p(\text{event} \mid x) = \sum_a \log \operatorname{sigmoid}\!\big(m_a(x) / v\big),$$

which is close to zero when every margin is positive and falls steeply as any margin goes negative. The softness $v$ sets how sharp that transition is.

`PreyBoom` uses $v = 0.01$. A much smaller $v$ makes guidance push hard until the margin is clearly positive, so it overshoots into implausibly large booms; a much larger $v$ lets the data outweigh the event once both are imposed. `event_rate` reports the fraction of paths meeting the event.

In [ ]:
def smooth_max(values: torch.Tensor, tau: float) -> torch.Tensor:
    """Log-sum-exp maximum over time: differentiable, and within tau * log(num_times) of the hard max."""
    return tau * torch.logsumexp(values / tau, dim=-1)


class PreyBoom(BinaryCondition):
    """Peak prey above `threshold`."""

    def __init__(self, threshold: float, v: float = 0.01, smooth_tau: float = 0.05):
        super().__init__(v=v)
        self.threshold = threshold
        self.smooth_tau = smooth_tau

    def smooth_eval_condition(self, x: torch.Tensor) -> torch.Tensor:
        margin = smooth_max(x[:, 0, :], self.smooth_tau) - self.threshold  # (B,)
        return margin[:, None, None]  # (B, 1, 1)


event_condition = PreyBoom(PEAK_PREY)


def event_rate(condition: BinaryCondition, samples: torch.Tensor) -> float:
    """Fraction of samples meeting every component of the event (all margins > 0)."""
    margins = condition.smooth_eval_condition(samples).squeeze(1)  # (B, components)
    return (margins.min(dim=-1).values > 0).double().mean().item()

## 4. Conditioned sampling

Each scenario wraps its condition in `GuidanceMC`, which estimates the guidance term by Monte Carlo, and samples with the same flow and sampler as the prior.

### 4.1 Conditioning on data

Sampling with the data condition alone pins down the first oscillation; the spread of the samples grows again after the observation window.

In [ ]:
data_guidance = GuidanceMC(flow, data_condition).to(device)
data_samples = sampler.sample(num_samples, guidance=data_guidance, num_steps=NUM_STEPS).detach()

fig, ax = plt.subplots(figsize=(6, 3.6))
plot_paths(ax, data_samples, "Conditioned on data")
plot_observations(ax, observations, [0, 1], obs_mask)
plt.tight_layout()

### 4.2 Conditioning on the event

Sampling with the prey-boom condition alone, with no data. The printed rates compare how often the boom occurs under the prior (from a large batch of prior paths) and under guidance.

In [ ]:
event_guidance = GuidanceMC(flow, event_condition).to(device)
event_samples = sampler.sample(num_samples, guidance=event_guidance, num_steps=NUM_STEPS).detach()

event_prior_samples = flow.map(torch.randn(20_000, 2, num_times, device=device))  # large prior batch for the rate
print(f"P(peak prey > {PEAK_PREY:g}): prior={event_rate(event_condition, event_prior_samples):.3f}  "
      f"guided={event_rate(event_condition, event_samples):.3f}")

fig, ax = plt.subplots(figsize=(6, 3.6))
plot_paths(ax, event_samples, "Conditioned on event")
show_event(ax, loc="upper right")
plt.tight_layout()

### 4.3 Conditioning on both

`ProductCondition` adds the two log-likelihoods, so guidance must fit the early observations **and** produce a later prey boom. The printout checks both: the event rate, and the mean absolute error at the observed points against the observation noise. These samples are plotted in the comparison below.

In [ ]:
both_guidance = GuidanceMC(flow, ProductCondition([data_condition, event_condition])).to(device)
both_samples = sampler.sample(num_samples, guidance=both_guidance, num_steps=NUM_STEPS).detach()

print(f"P(peak prey > {PEAK_PREY:g}) given both: {event_rate(event_condition, both_samples):.3f}")
obs_error = (both_samples[:, :, obs_mask] - truth[None, :, obs_mask]).abs().mean().item()
print(f"mean |error| at the observed points: {obs_error:.3f} (observation noise sd {OBS_NOISE})")

### 4.4 Side-by-side comparison

The four sets of samples side by side, as in Figure 4 of the LatentFlow paper. Dots mark the observations and dashed lines the event thresholds.

In [ ]:
panels = [
    (prior_samples, "Prior"),
    (data_samples, "Conditioned on data"),
    (event_samples, "Conditioned on event"),
    (both_samples, "Conditioned on both"),
]
fig, axes = plt.subplots(1, 4, figsize=(16, 3.6), sharex=True, sharey=True)
for i, (ax, (samples, panel_title)) in enumerate(zip(axes, panels, strict=False)):
    plot_paths(ax, samples, panel_title)
    if i:
        ax.set_ylabel(None)
    if i in (1, 3):
        plot_observations(ax, observations, [0, 1], obs_mask)
    if i in (2, 3):
        show_event(ax, loc="upper right")
channel_legend(axes[0], loc="upper left")
fig.suptitle("Stochastic Lotka–Volterra", y=1.03, fontsize=14, fontweight="bold")
plt.tight_layout()

## 5. Summary

- The stochastic Lotka–Volterra model is written as a map from Gaussian increments, so one flow and one sampler serve every scenario; only the condition changes.
- Conditioned on data, the samples follow the observations through the first oscillation and spread out again afterwards.
- Conditioned on the prey boom, guidance draws paths whose prey peak crosses the dashed threshold, and the predator responds through the coupled dynamics; compare the printed prior and guided event rates.
- `ProductCondition` combines the observations and the boom, giving paths that keep the observed start and bend the later dynamics towards the event.